# 10 · OCR cache cho ViTextVQA — chuẩn bị OCR-enhanced prompting (W8, bước 4)

**Vấn đề:** Error audit W7 cho thấy trên ViTextVQA, lỗi của B2 tập trung ở khâu đọc chữ: đọc chữ sai 28%, sai vùng 23%, sai dấu thanh 8%. Ablation "có / không OCR-enhanced prompting" lại là yêu cầu bắt buộc của đề cương. Để chạy được ablation này, mọi run (cả eval lẫn train) phải dùng **cùng một bản OCR**. Bản đó được tạo **một lần** ở notebook này và lưu thành file cache trên Drive.

**Notebook này làm 2 phần:**
- **Phần A (chạy ngay):** cài PaddleOCR, chạy thử 100 ảnh để đo tốc độ, rồi OCR **dev300** bằng **2 backend** (`paddleocr` và `paddledet_vietocr`). Cuối phần A có bảng chỉ số thay thế "đáp án có nằm trong OCR không".
- **Phần B (chạy SAU bước 5):** OCR **test2000** và **tập con train 8.000 câu** bằng backend đã chọn ở bước 5.
- **Phần C (W09, mục 11–12):** OCR phần ảnh còn thiếu của **full test** (`test.json`) bằng VietOCR, giữ nguyên phần cache cũ.
- **Phần D (W10, mục 13–14):** OCR phần ảnh còn thiếu của **toàn bộ train** (`train.json`) cho run B3-full, giữ nguyên phần cache cũ.

**Runtime:** chọn **T4 GPU** (Runtime → Change runtime type → T4 GPU). OCR không cần L4. Cache chỉ tạo một lần, mọi run sau đều đọc lại đúng file đó, nên GPU dùng để OCR không ảnh hưởng tính so sánh.

**Không cần** cài `transformers` hay nạp Qwen ở đây. Thư viện OCR chạy trong phiên riêng này để không đụng tới môi trường chạy VLM.

## 1. Thiết lập môi trường

Gắn Google Drive, lấy code mới nhất từ GitHub, in loại GPU.

*Kỳ vọng: commit mới nhất (có `src/eval/ocr_coverage.py`), `GPU: Tesla T4` (hoặc GPU khác, đều được), `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('src/eval/ocr_coverage.py'), 'Code cũ -> git pull chưa lấy được commit W08 bước 4'

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU (OCR sẽ chạy CPU, chậm hơn)')
print('✅ Thiết lập xong')

## 2. Cài PaddleOCR 3.7 (thử GPU trước, không được thì tự chuyển CPU) — **torch phải còn nguyên**

**Vấn đề (gặp thật ở lần chạy 13/10):** `paddlepaddle-gpu` bản CUDA 12.6 cài kèm thư viện NCCL của CUDA 12, **ghi đè lên file NCCL mà torch của Colab (CUDA 13) đang dùng**. Kết quả là `import torch` hỏng (`undefined symbol: ncclCommWindowDeregister`). Mà `paddleocr` lại import torch qua `modelscope`, nên OCR cũng hỏng theo.

**Cách cell này xử lý:**
1. Kiểm tra torch còn tốt **trước khi** cài gì, và chụp danh sách thư viện `nvidia-*` của torch.
2. Cài `paddleocr==3.7.0` (ghim phiên bản, vì code viết theo API 3.x), rồi thử `paddlepaddle-gpu`.
3. **Cài lại đúng bản NCCL của torch.** Nếu torch vẫn hỏng thì cài lại toàn bộ thư viện `nvidia-*` theo danh sách đã chụp.
4. Kiểm tra ở **tiến trình riêng**: torch thấy GPU **và** Paddle chạy được trên GPU → `DEVICE = gpu`. Nếu không, gỡ Paddle GPU, khôi phục thư viện của torch, cài Paddle CPU → `DEVICE = cpu` (chậm hơn nhưng chắc chắn chạy).
5. Cuối cell **bắt buộc torch còn tốt**, nếu không thì dừng hẳn.

*Kỳ vọng: `torch OK trước khi cài`, phiên bản `paddleocr 3.7.0 | paddle …`, rồi `DEVICE = gpu` (hoặc `cpu` kèm lý do) và `✅ torch vẫn tốt`. Mất khoảng 3–6 phút.*

In [ ]:
import subprocess, json

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-1500:]

def nvidia_pkgs():
    out = subprocess.run('pip list --format=json', shell=True, capture_output=True, text=True).stdout
    return {p['name'].lower(): p['version'] for p in json.loads(out) if p['name'].lower().startswith('nvidia-')}

def torch_ok():
    return sh('python -c "import torch; assert torch.cuda.is_available(); torch.zeros(1).cuda()"')[0] == 0

def paddle_gpu_ok():
    return sh('python -c "import paddle; assert paddle.device.is_compiled_with_cuda(); paddle.utils.run_check()"')[0] == 0

def restore(pkgs):                       # cài lại ĐÚNG phiên bản cũ, không kéo thêm gì
    if pkgs:
        code_, log = sh('pip install -q --force-reinstall --no-deps ' + ' '.join(f'{n}=={v}' for n, v in pkgs.items()))
        assert code_ == 0, log

assert torch_ok(), '❌ torch đã hỏng từ trước -> Runtime -> Disconnect and delete runtime, rồi chạy lại từ mục 1'
print('torch OK trước khi cài')
code_, log = sh('pip install -q paddleocr==3.7.0')
assert code_ == 0, log
BEFORE = nvidia_pkgs()                   # thư viện CUDA của torch (chụp lại để khôi phục)

code_gpu, log_gpu = sh('pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/')
NEW = sorted(set(nvidia_pkgs()) - set(BEFORE))
restore({n: v for n, v in BEFORE.items() if 'nccl' in n})     # NCCL của torch bị ghi đè -> cài lại
if not torch_ok():
    print('torch vẫn hỏng sau khi khôi phục NCCL -> khôi phục toàn bộ thư viện nvidia-* của torch')
    restore(BEFORE)

if code_gpu == 0 and torch_ok() and paddle_gpu_ok():
    DEVICE = 'gpu'
else:
    print('⚠ Paddle GPU không chạy chung được với torch' if code_gpu == 0 else '⚠ Cài Paddle GPU lỗi:\n' + log_gpu[-400:])
    print('-> gỡ Paddle GPU, khôi phục thư viện torch, cài Paddle CPU')
    sh('pip uninstall -y -q paddlepaddle-gpu ' + ' '.join(NEW))
    restore(BEFORE)
    code_, log = sh('pip install -q paddlepaddle')
    assert code_ == 0, log
    DEVICE = 'cpu'

assert torch_ok(), '❌ torch hỏng sau khi cài Paddle -> gửi toàn bộ output cell này cho Claude'
print(sh('python -c "import paddleocr, paddle; print(\'paddleocr\', paddleocr.__version__, \'| paddle\', paddle.__version__)"')[1])
print('DEVICE =', DEVICE, '| ✅ torch vẫn tốt')

## 3. Ảnh + danh sách ảnh cần OCR

Giải nén ảnh ViTextVQA từ zip trên Drive về `/content`, giống notebook 02/08. Sau đó đếm số **ảnh duy nhất** của từng file dữ liệu: nhiều câu hỏi có thể dùng chung một ảnh, và OCR chỉ cần chạy một lần cho mỗi ảnh.

*Kỳ vọng: bảng 3 dòng `dev300`, `test2000`, `train_sub8k`. Dòng `train_sub8k` phải có **2670** ảnh (khớp manifest bước 3). Không có dòng `⚠ thiếu`. Lần đầu giải nén mất khoảng 3–5 phút.*

In [ ]:
from src.eval.run_baseline import prepare_images
from src.data.ocr import images_of, default_cache_path

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
DS = 'vitextvqa_official'
DATA_DIR = f'{DRIVE_ROOT}/data/{DS}'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
IMG = prepare_images(DS, DRIVE_ROOT)

FILES = {'dev300': 'dev_subset300_seed42.json',
         'test2000': 'test_subset2000_seed42.json',
         'train_sub8k': 'train_subset8000_byimage_seed42.json'}
IMGS = {}
for k, f in FILES.items():
    IMGS[k] = images_of([f'{DATA_DIR}/{f}'], IMG)
    miss = [p for p in IMGS[k].values() if not os.path.exists(p)]
    print(f'{k:12s} {len(IMGS[k]):5d} ảnh' + (f'  ⚠ thiếu {len(miss)} ảnh' if miss else ''))
assert len(IMGS['train_sub8k']) == 2670, 'Tập con train không khớp manifest bước 3'
ALL_B = set(IMGS['test2000']) | set(IMGS['train_sub8k'])
print(f'Phần B (test2000 ∪ train_sub8k): {len(ALL_B)} ảnh')

## 4. Chạy thử 100 ảnh (backend `paddleocr`) → đo tốc độ, xem tận mắt

Chỉ OCR 100 ảnh đầu của dev300, rồi:
- tính **giây/ảnh** và **thời gian ước tính** cho phần còn lại;
- in 3 câu hỏi kèm đáp án gold và các dòng OCR đọc được, để **xem tận mắt** OCR có đọc ra chữ tiếng Việt có dấu không.

Ảnh ViTextVQA đều có chữ, nên nếu dưới 50% ảnh đọc ra chữ thì gần như chắc chắn có lỗi cài đặt. Khi đó cell sẽ dừng.

*Kỳ vọng: lần đầu tải model OCR (vài chục MB). Sau đó `[OCR] xong … s/ảnh`. GPU khoảng 0,1–0,5 s/ảnh, CPU khoảng 1–4 s/ảnh. Tỉ lệ ảnh có chữ ≥ 50%, ví dụ in ra có dấu tiếng Việt.*

In [ ]:
import json
from src.data.ocr import load_ocr_cache

C_PADDLE = default_cache_path(DRIVE_ROOT, DS, 'paddleocr')
!python -u -m src.data.ocr --dataset {DS} --backend paddleocr --device {DEVICE} --image_dir {IMG} --files dev_subset300_seed42.json --limit_images 100

c = load_ocr_cache(C_PADDLE)
assert c['images'] and c['timing'].get('images'), '❌ OCR chưa chạy được ảnh nào (xem lỗi phía trên) -> gửi output cell này cho Claude'
t = c['timing']; spi = t['sec'] / t['images']
with_text = sum(bool(e['lines']) for e in c['images'].values()) / len(c['images'])
print(f"\nengine: {c['engine_info']}")
print(f'{spi:.2f} s/ảnh | {100 * with_text:.0f}% ảnh có chữ')
print(f'Ước tính: phần còn lại dev300 ≈ {spi * (len(IMGS["dev300"]) - t["images"]) / 60:.0f} phút | '
      f'phần B ({len(ALL_B)} ảnh) ≈ {spi * len(ALL_B) / 60:.0f} phút')
assert with_text >= 0.5, 'Dưới 50% ảnh có chữ -> OCR cài sai? Dán log cell này cho Claude'

dev = json.load(open(f'{DATA_DIR}/{FILES["dev300"]}', encoding='utf-8'))
shown = 0
for r in dev:
    e = c['images'].get(r['image'])
    if e and shown < 3:
        shown += 1
        print(f"\n— {r['question']}  | gold: {r['answers']}")
        for ln in e['lines'][:8]:
            print(f"   {ln['conf']:.2f}  {ln['text']}")

## 5. OCR hết dev300 bằng `paddleocr`

Chạy tiếp từ chỗ dừng: 100 ảnh đã OCR ở mục 4 được giữ nguyên, không chạy lại. Nếu Colab bị ngắt giữa chừng thì chạy lại đúng cell này, vì cache được lưu sau mỗi 50 ảnh.

*Kỳ vọng: `[OCR] … sẽ OCR <còn lại>` rồi `[OCR] xong. Cache: …/ocr/paddleocr.json`.*

In [ ]:
!python -u -m src.data.ocr --dataset {DS} --backend paddleocr --device {DEVICE} --image_dir {IMG} --files dev_subset300_seed42.json

## 6. Backend thứ hai: PaddleOCR dò chữ + VietOCR đọc chữ (dev300)

VietOCR là mô hình đọc chữ **huấn luyện riêng cho tiếng Việt**. Backend này dùng **cùng model dò chữ** với `paddleocr` và chỉ thay phần đọc, nên so hai backend là so đúng một biến: phần đọc chữ.

Cài `vietocr` với `--no-deps`, vì gói trên PyPI khoá `pillow==10.2.0` và sẽ hạ cấp Pillow của Colab. Phần đọc chữ chỉ cần thêm `gdown` và `einops`.

*Kỳ vọng: `VietOCR import OK`, lần đầu tải trọng số VietOCR, rồi `[OCR] xong. Cache: …/ocr/paddledet_vietocr.json`. Chậm hơn `paddleocr` vì đọc từng hộp chữ.*

In [ ]:
code_, log = sh('pip install -q --no-deps vietocr==0.3.13 && pip install -q gdown einops')
assert code_ == 0, log
print(sh('python -c "from vietocr.tool.predictor import Predictor; print(\'VietOCR import OK\')"')[1])
C_VIET = default_cache_path(DRIVE_ROOT, DS, 'paddledet_vietocr')
!python -u -m src.data.ocr --dataset {DS} --backend paddledet_vietocr --device {DEVICE} --image_dir {IMG} --files dev_subset300_seed42.json

## 7. Chỉ số thay thế trên dev300: đáp án gold có nằm trong văn bản OCR không?

Chưa có nhãn chữ thật nên không tính được tỉ lệ lỗi ký tự của OCR. Thay vào đó, mỗi backend được đo trên 300 câu dev:
- **đáp án∈OCR %**: đáp án gold xuất hiện nguyên cụm từ trong văn bản OCR. Đây là **trần trên** cho lợi ích của OCR-enhanced prompting.
- **…sau cắt %**: như trên nhưng chỉ tính phần thực sự vào prompt (cắt ở 600 ký tự).
- **…bỏ dấu %**: bỏ dấu cả hai phía. Phần chênh so với cột đầu là phần OCR **đọc đúng chữ nhưng sai dấu thanh**.
- **Ghép cặp McNemar**: hai backend khác nhau có ý nghĩa thống kê không.

Đây chỉ là **chẩn đoán**. Backend được chọn theo luật đặt trước ở bước 5 (EM của B1 + OCR trên dev300).

*Kỳ vọng: bảng 2 dòng + 1 dòng `Ghép cặp …`, ghi ra `experiments/W08_ocr_dev300_coverage/ocr_coverage.json`.*

In [ ]:
OUT7 = f'{EXP_ROOT}/W08_ocr_dev300_coverage/ocr_coverage.json'
!python -m src.eval.ocr_coverage --data {DATA_DIR}/dev_subset300_seed42.json --caches {C_PADDLE} {C_VIET} --out {OUT7}

## 8. ⏸ DỪNG — hết Phần A

Gửi Claude: **ảnh chụp output mục 4 và mục 7** (hoặc copy chữ). Sau đó **Disconnect and delete runtime** để không tốn thêm compute.

Bước 5 (phiên L4 khác, notebook riêng) sẽ chạy B1 + OCR trên dev300 cho từng backend và template để chọn cấu hình theo luật đặt trước. Có kết quả bước 5 rồi mới quay lại chạy **Phần B** bên dưới.

## 9. PHẦN B — OCR test2000 + tập con train bằng backend đã chọn

Điền `BACKEND` theo kết quả bước 5 (`'paddleocr'` hoặc `'paddledet_vietocr'`). Khi chạy Phần B trong **phiên mới**, chạy lại mục 1 → 3 trước, rồi chạy mục 6 nếu backend là `paddledet_vietocr`. Cache dev300 đã có trên Drive nên không phải chạy lại mục 4–7.

*Kỳ vọng: 2 lượt `[OCR] xong`, thời gian gần với ước tính ở mục 4. Chạy lại khi bị ngắt sẽ tự tiếp từ chỗ dừng.*

In [ ]:
BACKEND = None          # <- điền sau bước 5, vd 'paddleocr'
assert BACKEND in ('paddleocr', 'paddledet_vietocr'), 'Điền BACKEND theo kết quả bước 5'
!python -u -m src.data.ocr --dataset {DS} --backend {BACKEND} --device {DEVICE} --image_dir {IMG} --files test_subset2000_seed42.json
!python -u -m src.data.ocr --dataset {DS} --backend {BACKEND} --device {DEVICE} --image_dir {IMG} --files train_subset8000_byimage_seed42.json

## 10. Biên bản cache (để tái lập)

Ghi sha256 của từng file cache, engine, số ảnh và thời gian OCR ra `experiments/W08_ocr_cache/ocr_cache_manifest.json`. File này nhỏ, sẽ được đưa lên git. Bản thân các file cache nằm trên Drive.

*Kỳ vọng: mỗi cache 1 dòng (backend, số ảnh, s/ảnh, sha256 12 ký tự đầu). Tập ảnh dev300/test2000/train_sub8k đều đủ với backend đã chọn.*

In [ ]:
import hashlib, datetime
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

man = {'what': 'OCR cache ViTextVQA (W08) — file cache nằm trên Drive, đây là biên bản để tái lập',
       'created_utc': datetime.datetime.utcnow().isoformat(timespec='seconds'), 'caches': []}
for p in [C_PADDLE, C_VIET]:
    if not os.path.exists(p):
        continue
    c = load_ocr_cache(p)
    cover = {k: sum(x in c['images'] for x in v) for k, v in IMGS.items()}
    man['caches'].append({'file': p.replace(DRIVE_ROOT + '/', ''), 'sha256': sha256(p), 'backend': c['backend'],
                          'engine_info': c['engine_info'], 'n_images': len(c['images']),
                          'n_errors': sum(1 for e in c['images'].values() if e.get('error')),
                          'timing': c['timing'], 'images_covered': cover,
                          'images_needed': {k: len(v) for k, v in IMGS.items()}})
    print(f"{c['backend']:18s} {len(c['images']):5d} ảnh | {c['timing']['sec'] / c['timing']['images']:.2f} s/ảnh | "
          f"sha256 {man['caches'][-1]['sha256'][:12]} | đủ: {cover}")
os.makedirs(f'{EXP_ROOT}/W08_ocr_cache', exist_ok=True)
json.dump(man, open(f'{EXP_ROOT}/W08_ocr_cache/ocr_cache_manifest.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('->', f'{EXP_ROOT}/W08_ocr_cache/ocr_cache_manifest.json')

## 11. PHẦN C (W09) — OCR phần ảnh còn thiếu của full test (`test.json`)

**Vì sao:** W09 chạy B3 và B2-sub trên **full test 10.028 câu / 3.353 ảnh**. Cache VietOCR mới có 1.486 ảnh test (của test2000), còn thiếu ≈ 1.867 ảnh.

**Cách chạy (phiên mới, T4):** chạy **mục 1 → 3** (thiết lập, cài Paddle, giải nén ảnh), rồi **nhảy thẳng tới mục 11**. Không cần chạy mục 4–10. Cell này tự cài VietOCR.

**Bảo vệ cache cũ (luật W09):**
- Lần chạy đầu: cache trên Drive phải **trùng sha256** với biên bản W08 (`experiments/W08_ocr_cache/`). Khi đó ghi lại dấu vân tay của 4.422 ảnh đã có vào `W09_ocr_cache_fulltest/pre_extension.json` trên Drive.
- Sau khi OCR thêm: dấu vân tay của đúng 4.422 ảnh đó phải **giữ nguyên** → mọi run W08/W09 đã chấm không bị ảnh hưởng.
- Bị ngắt giữa chừng: chạy lại cell, cache tự chạy tiếp (lưu mỗi 50 ảnh).

*Kỳ vọng: `cache = biên bản W08 ✅` (lần đầu) hoặc `đã có pre_extension.json` (chạy lại), `[OCR] … sẽ OCR ~1867`, `[OCR] xong`, `phần cache cũ giữ nguyên ✅`, `full test 3353/3353 ✅`. GPU T4 ≈ 30–40 phút; nếu `DEVICE = cpu` thì chậm hơn nhiều.*

In [ ]:
import hashlib
from src.data.ocr import load_ocr_cache, cache_fingerprint

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

code_, log = sh('pip install -q --no-deps vietocr==0.3.13 && pip install -q gdown einops')
assert code_ == 0, log
BACKEND_C = json.load(open('experiments/W08_ocr_select_dev300_L4_seed42/ocr_selection.json', encoding='utf-8'))['decision']['backend']
assert BACKEND_C == 'paddledet_vietocr', BACKEND_C
C_PATH = default_cache_path(DRIVE_ROOT, DS, BACKEND_C)
W09_OCR_DIR = f'{EXP_ROOT}/W09_ocr_cache_fulltest'
PRE = f'{W09_OCR_DIR}/pre_extension.json'
os.makedirs(W09_OCR_DIR, exist_ok=True)

if not os.path.exists(PRE):
    w08 = json.load(open('experiments/W08_ocr_cache/ocr_cache_manifest.json', encoding='utf-8'))
    want = next(c['sha256'] for c in w08['caches'] if c['backend'] == BACKEND_C)
    assert sha256_file(C_PATH) == want, '❌ Cache trên Drive đã khác biên bản W08 mà chưa có pre_extension.json -> DỪNG, báo Claude'
    old = load_ocr_cache(C_PATH)
    keys = sorted(old['images'])
    json.dump({'cache_sha256_w08': want, 'n_images': len(keys), 'fingerprint': cache_fingerprint(old, keys),
               'engine_info': old['engine_info'], 'keys': keys},
              open(PRE, 'w', encoding='utf-8'), ensure_ascii=False, indent=0)
    print(f'cache = biên bản W08 ✅ ({len(keys)} ảnh) -> ghi {PRE}')
else:
    print('đã có pre_extension.json (chạy lại) -> dùng dấu vân tay đã ghi')
pre = json.load(open(PRE, encoding='utf-8'))

FULL_IMGS = images_of([f'{DATA_DIR}/test.json'], IMG)
print(f"full test: {len(FULL_IMGS)} ảnh | đã có trong cache {len(set(FULL_IMGS) & set(load_ocr_cache(C_PATH)['images']))}")
!python -u -m src.data.ocr --dataset {DS} --backend {BACKEND_C} --device {DEVICE} --image_dir {IMG} --files test.json

after = load_ocr_cache(C_PATH)
assert set(pre['keys']) <= set(after['images']) and cache_fingerprint(after, pre['keys']) == pre['fingerprint'], \
    '❌ Phần cache cũ BỊ ĐỔI -> DỪNG, báo Claude'
print('phần cache cũ giữ nguyên ✅')
miss = set(FULL_IMGS) - set(after['images'])
added = sorted(set(after['images']) - set(pre['keys']))
n_err = sum(1 for k in added if after['images'][k].get('error'))
print(f"full test {len(FULL_IMGS) - len(miss)}/{len(FULL_IMGS)} {'✅' if not miss else '❌'} | ảnh mới {len(added)} | lỗi {n_err}")
assert not miss and n_err <= 0.01 * max(len(added), 1), 'Thiếu ảnh hoặc lỗi > 1% -> gửi output cho Claude'
for k in ('paddleocr', 'paddlepaddle', 'det_model', 'rec_model', 'vietocr_config', 'device'):
    o, n = pre['engine_info'].get(k), after['engine_info'].get(k)
    if o != n:
        print(f'⚠ engine_info.{k}: W08 = {o} | đợt W09 = {n} (ghi vào biên bản)')

## 12. Biên bản cache W09 (để tái lập)

Ghi `experiments/W09_ocr_cache_fulltest/ocr_cache_manifest.json`: sha256 cache mới, số ảnh, số ảnh thêm, engine của đợt W08 và đợt W09, dấu vân tay phần cũ. File nhỏ, Cursor chép lên git (**không** chép `pre_extension.json`).

*Kỳ vọng: 1 dòng tóm tắt + đường dẫn biên bản. Sau đó **Runtime → Disconnect and delete runtime**, rồi mở notebook 15 trên L4.*

In [ ]:
import datetime
man = {'what': 'OCR cache VietOCR mở rộng cho ViTextVQA full test (W09) — file cache nằm trên Drive',
       'created_utc': datetime.datetime.utcnow().isoformat(timespec='seconds'),
       'cache_file': C_PATH.replace(DRIVE_ROOT + '/', ''), 'sha256_before_w08': pre['cache_sha256_w08'],
       'sha256_after_w09': sha256_file(C_PATH), 'n_images_before': pre['n_images'], 'n_images_after': len(after['images']),
       'n_images_added': len(added), 'n_errors_added': n_err, 'full_test_images': len(FULL_IMGS),
       'old_part_fingerprint': pre['fingerprint'], 'old_part_unchanged': True,
       'engine_info_w08': pre['engine_info'], 'engine_info_w09': after['engine_info'], 'device_w09': DEVICE,
       'timing_total': after['timing']}
json.dump(man, open(f'{W09_OCR_DIR}/ocr_cache_manifest.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(f"{man['n_images_before']} -> {man['n_images_after']} ảnh (+{man['n_images_added']}, lỗi {n_err}) | "
      f"sha256 {man['sha256_after_w09'][:12]} | device {DEVICE}")
print('->', f'{W09_OCR_DIR}/ocr_cache_manifest.json')

## 13. PHẦN D (W10) — OCR phần ảnh còn thiếu của toàn bộ train (`train.json`)

**Vì sao:** W10 chạy **B3-full** = công thức B3 trên **toàn bộ 35.159 câu / 11.733 ảnh train** (notebook 18). Cache VietOCR mới có 2.670 ảnh train (của tập con 8k), còn thiếu ≈ 9.063 ảnh.

**Cách chạy (phiên mới, T4):** chạy **mục 1 → 3** (thiết lập, cài Paddle, giải nén ảnh), rồi **nhảy thẳng tới mục 13**. Không cần chạy mục 4–12. Cell này tự cài VietOCR.

**Bảo vệ cache cũ (luật B3-full, `docs/W10/README.md`):**
- Lần chạy đầu: cache trên Drive phải **trùng sha256** với biên bản W09 (`experiments/W09_ocr_cache_fulltest/`, 6.289 ảnh). Khi đó ghi dấu vân tay của 6.289 ảnh đã có vào `W10_ocr_cache_fulltrain/pre_extension.json` trên Drive.
- Sau khi OCR thêm: dấu vân tay của đúng 6.289 ảnh đó phải **giữ nguyên** → mọi run W08–W10 đã chấm không bị ảnh hưởng.
- Bị ngắt giữa chừng: chạy lại cell này, cache tự chạy tiếp (lưu mỗi 50 ảnh).

*Kỳ vọng: `cache = biên bản W09 ✅ (6289 ảnh)` (lần đầu) hoặc `đã có pre_extension.json` (chạy lại), `train: 11733 ảnh | đã có trong cache 2670`, `[OCR] … sẽ OCR ~9063`, `[OCR] xong`, `phần cache cũ giữ nguyên ✅`, `train 11733/11733 ✅`, `full test 3353/3353 ✅`. GPU T4 ≈ 2–2,5 giờ; nếu `DEVICE = cpu` thì chậm hơn nhiều → báo Claude trước khi chạy tiếp.*

In [ ]:
import hashlib
from src.data.ocr import load_ocr_cache, cache_fingerprint

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

code_, log = sh('pip install -q --no-deps vietocr==0.3.13 && pip install -q gdown einops')
assert code_ == 0, log
BACKEND_D = json.load(open('experiments/W08_ocr_select_dev300_L4_seed42/ocr_selection.json', encoding='utf-8'))['decision']['backend']
assert BACKEND_D == 'paddledet_vietocr', BACKEND_D
C_PATH = default_cache_path(DRIVE_ROOT, DS, BACKEND_D)
W10_OCR_DIR = f'{EXP_ROOT}/W10_ocr_cache_fulltrain'
PRE_D = f'{W10_OCR_DIR}/pre_extension.json'
os.makedirs(W10_OCR_DIR, exist_ok=True)

if not os.path.exists(PRE_D):
    w09 = json.load(open('experiments/W09_ocr_cache_fulltest/ocr_cache_manifest.json', encoding='utf-8'))
    want = w09['sha256_after_w09']
    assert sha256_file(C_PATH) == want, '❌ Cache trên Drive đã khác biên bản W09 mà chưa có pre_extension.json -> DỪNG, báo Claude'
    old = load_ocr_cache(C_PATH)
    keys = sorted(old['images'])
    assert len(keys) == w09['n_images_after'], (len(keys), w09['n_images_after'])
    json.dump({'cache_sha256_w09': want, 'n_images': len(keys), 'fingerprint': cache_fingerprint(old, keys),
               'engine_info': old['engine_info'], 'keys': keys},
              open(PRE_D, 'w', encoding='utf-8'), ensure_ascii=False, indent=0)
    print(f'cache = biên bản W09 ✅ ({len(keys)} ảnh) -> ghi {PRE_D}')
else:
    print('đã có pre_extension.json (chạy lại) -> dùng dấu vân tay đã ghi')
pre_d = json.load(open(PRE_D, encoding='utf-8'))

TRAIN_IMGS = images_of([f'{DATA_DIR}/train.json'], IMG)
miss_disk = [p for p in TRAIN_IMGS.values() if not os.path.exists(p)]
assert not miss_disk, f'Thiếu {len(miss_disk)} ảnh train trên đĩa (vd {miss_disk[:3]}) -> chạy lại mục 3'
print(f"train: {len(TRAIN_IMGS)} ảnh | đã có trong cache {len(set(TRAIN_IMGS) & set(load_ocr_cache(C_PATH)['images']))}")
!python -u -m src.data.ocr --dataset {DS} --backend {BACKEND_D} --device {DEVICE} --image_dir {IMG} --files train.json

after_d = load_ocr_cache(C_PATH)
assert set(pre_d['keys']) <= set(after_d['images']) and cache_fingerprint(after_d, pre_d['keys']) == pre_d['fingerprint'], \
    '❌ Phần cache cũ BỊ ĐỔI -> DỪNG, báo Claude'
print('phần cache cũ giữ nguyên ✅')
FULL_TEST_IMGS = images_of([f'{DATA_DIR}/test.json'], IMG)
miss_tr = set(TRAIN_IMGS) - set(after_d['images'])
miss_te = set(FULL_TEST_IMGS) - set(after_d['images'])
added_d = sorted(set(after_d['images']) - set(pre_d['keys']))
n_err_d = sum(1 for k in added_d if after_d['images'][k].get('error'))
with_text_d = sum(bool(after_d['images'][k].get('lines')) for k in added_d) / max(len(added_d), 1)
print(f"train {len(TRAIN_IMGS) - len(miss_tr)}/{len(TRAIN_IMGS)} {'✅' if not miss_tr else '❌'} | "
      f"full test {len(FULL_TEST_IMGS) - len(miss_te)}/{len(FULL_TEST_IMGS)} {'✅' if not miss_te else '❌'} | "
      f"ảnh mới {len(added_d)} | lỗi {n_err_d} | {100 * with_text_d:.1f}% ảnh mới có chữ")
assert not miss_tr and not miss_te and n_err_d <= 0.01 * max(len(added_d), 1), 'Thiếu ảnh hoặc lỗi > 1% -> gửi output cho Claude'
for k in ('paddleocr', 'paddlepaddle', 'det_model', 'rec_model', 'vietocr_config', 'device'):
    o, n = pre_d['engine_info'].get(k), after_d['engine_info'].get(k)
    if o != n:
        print(f'⚠ engine_info.{k}: trước = {o} | đợt W10 = {n} (ghi vào biên bản)')

## 14. Biên bản cache W10 (để tái lập)

Ghi `experiments/W10_ocr_cache_fulltrain/ocr_cache_manifest.json`: sha256 cache mới, số ảnh, số ảnh thêm, engine trước và đợt W10, dấu vân tay phần cũ. File nhỏ, Cursor chép lên git (**không** chép `pre_extension.json`). Notebook 18 kiểm sha256 cache trên Drive khớp biên bản này trước khi train.

*Kỳ vọng: 1 dòng tóm tắt (≈ 6289 -> 15352 ảnh) + đường dẫn biên bản. Sau đó **Runtime → Disconnect and delete runtime**, rồi mở notebook 18 trên **L4**.*

In [ ]:
import datetime
man_d = {'what': 'OCR cache VietOCR mở rộng cho ViTextVQA TOÀN BỘ train (W10, cho B3-full) — file cache nằm trên Drive',
         'created_utc': datetime.datetime.utcnow().isoformat(timespec='seconds'),
         'cache_file': C_PATH.replace(DRIVE_ROOT + '/', ''), 'sha256_before_w10': pre_d['cache_sha256_w09'],
         'sha256_after_w10': sha256_file(C_PATH), 'n_images_before': pre_d['n_images'], 'n_images_after': len(after_d['images']),
         'n_images_added': len(added_d), 'n_errors_added': n_err_d, 'pct_added_with_text': round(100 * with_text_d, 2),
         'train_images': len(TRAIN_IMGS), 'full_test_images': len(FULL_TEST_IMGS),
         'old_part_fingerprint': pre_d['fingerprint'], 'old_part_unchanged': True,
         'engine_info_before': pre_d['engine_info'], 'engine_info_w10': after_d['engine_info'], 'device_w10': DEVICE,
         'timing_total': after_d['timing']}
json.dump(man_d, open(f'{W10_OCR_DIR}/ocr_cache_manifest.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(f"{man_d['n_images_before']} -> {man_d['n_images_after']} ảnh (+{man_d['n_images_added']}, lỗi {n_err_d}) | "
      f"sha256 {man_d['sha256_after_w10'][:12]} | device {DEVICE}")
print('->', f'{W10_OCR_DIR}/ocr_cache_manifest.json')